In [3]:
import os
import glob
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# ---------- 1) Find dataset paths ----------
if not os.path.exists("/kaggle/input"):
    raise FileNotFoundError(
        "Не найден /kaggle/input. Это не Kaggle Notebook или данные не подключены."
    )

# Prefer the standard Titanic competition folder if it exists
preferred_train = "/kaggle/input/titanic/train.csv"
preferred_test  = "/kaggle/input/titanic/test.csv"

if os.path.exists(preferred_train) and os.path.exists(preferred_test):
    train_path, test_path = preferred_train, preferred_test
else:
    train_candidates = glob.glob("/kaggle/input/**/train.csv", recursive=True)
    test_candidates  = glob.glob("/kaggle/input/**/test.csv", recursive=True)

    # try to pick ones that contain "titanic" in the path
    train_titanic = [p for p in train_candidates if "titanic" in p.lower()]
    test_titanic  = [p for p in test_candidates if "titanic" in p.lower()]

    if train_titanic and test_titanic:
        train_path, test_path = train_titanic[0], test_titanic[0]
    elif train_candidates and test_candidates:
        train_path, test_path = train_candidates[0], test_candidates[0]
    else:
        raise FileNotFoundError(
            "Не нашёл train.csv/test.csv в /kaggle/input. "
            "Нужно добавить данные соревнования Titanic в ноутбук (Add Data)."
        )

print("Using train:", train_path)
print("Using test: ", test_path)

# ---------- 2) Load data ----------
train_data = pd.read_csv(train_path)
test_data  = pd.read_csv(test_path)

print("Train shape:", train_data.shape)
print("Test shape: ", test_data.shape)

# ---------- 3) Prepare features/target ----------
y = train_data["Survived"]

features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
X = train_data[features]
X_test = test_data[features]

numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Sex", "Embarked"]

numeric_transformer = SimpleImputer(strategy="median")
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# ---------- 4) Model ----------
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    random_state=42
)

clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model)
])

# Fit on full training data
clf.fit(X, y)

# Predict for test
predictions = clf.predict(X_test)

# ---------- 5) Save submission ----------
submission = pd.DataFrame({
    "PassengerId": test_data["PassengerId"],
    "Survived": predictions.astype(int)
})
submission.to_csv("submission.csv", index=False)

print("OK: saved submission.csv")
print(submission.head(10))

Using train: /kaggle/input/competitions/titanic/train.csv
Using test:  /kaggle/input/competitions/titanic/test.csv
Train shape: (891, 12)
Test shape:  (418, 11)
OK: saved submission.csv
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
5          897         0
6          898         1
7          899         0
8          900         1
9          901         0
